# TP — Séance 2 : RAG, construire un assistant documentaire
### Master IDD · IA générative & LLM — IAE Grenoble

Ce notebook accompagne la Séance 2 du cours. Il contient les **trois TP** de la séance :

1. **TP1 — Embeddings et recherche sémantique** (~25 min)
2. **TP2 — Construire un pipeline RAG** (~45 min)
3. **TP3 — Évaluer le RAG** (~25 min)

**Outils utilisés (100% gratuits) :** Google Colab (GPU), un LLM open-source exécuté en local avec `transformers`, ChromaDB, `sentence-transformers`, LangChain (découpage de texte), `pypdf`, `pandas`.

> 💡 Ouvrez ce notebook dans **Google Colab** et activez le GPU (Exécution > Modifier le type d'exécution > GPU T4). Aucun token ni compte n'est nécessaire.

## 0. Préparation

Aucun token ni secret n'est nécessaire : le **LLM** et le **modèle d'embeddings** sont téléchargés et exécutés directement sur Colab.

Les cellules suivantes installent les bibliothèques (1 à 2 minutes), téléchargent le LLM, puis préparent le modèle d'embeddings.

In [ ]:
!pip install -q chromadb sentence-transformers langchain-text-splitters pypdf pandas transformers accelerate

In [ ]:
import time
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Modèles ouverts exécutés en local sur Colab (aucune API, aucun token)
OPEN_MODELS = {
    "Qwen 2.5 3B Instruct": "Qwen/Qwen2.5-3B-Instruct",
    "Qwen 2.5 1.5B Instruct": "Qwen/Qwen2.5-1.5B-Instruct",
    "Phi 3.5 mini Instruct": "microsoft/Phi-3.5-mini-instruct",
}
MODELE_LLM = OPEN_MODELS["Qwen 2.5 3B Instruct"]

# Modèle d'embeddings multilingue (gratuit, exécuté dans Colab)
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="intfloat/multilingual-e5-small"
)
chroma_client = chromadb.Client()

In [ ]:
# Téléchargement du LLM sur Colab (quelques Go, une seule fois par session)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Téléchargement de {MODELE_LLM} (device : {DEVICE})...")
tokenizer = AutoTokenizer.from_pretrained(MODELE_LLM)
llm = AutoModelForCausalLM.from_pretrained(
    MODELE_LLM,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def ask_model(prompt, max_tokens=400, system=None):
    """Interroge le LLM local chargé ci-dessus et renvoie sa réponse."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(llm.device)
    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print("Configuration prête.")

---
## TP1 — Embeddings et recherche sémantique (~25 min)

**Objectif :** vectoriser un petit corpus, l'indexer dans une base vectorielle (ChromaDB) et observer comment la **formulation de la question** influence les passages retrouvés.

In [ ]:
# 1.1 — Mini-corpus fictif : règlement interne de l'entreprise « Alpina Conseil »
corpus = [
    ("Congés payés : chaque salarié acquiert 2,5 jours ouvrés de congés par mois travaillé, soit 30 jours par an.", "reglement_rh"),
    ("Télétravail : il est possible de travailler à distance jusqu'à deux jours par semaine, après accord du manager.", "reglement_rh"),
    ("Frais de repas : les repas d'affaires sont remboursés sur présentation d'un justificatif, dans la limite de 25 euros par personne.", "notes_de_frais"),
    ("Déplacements : les billets de train en seconde classe sont remboursés ; la première classe nécessite l'accord de la direction.", "notes_de_frais"),
    ("Préavis : en cas de démission, la durée du préavis est de trois mois pour les cadres et de un mois pour les autres salariés.", "reglement_rh"),
    ("Formation : chaque salarié dispose de 3 jours de formation par an, à choisir dans le catalogue interne.", "reglement_rh"),
    ("Sécurité informatique : les mots de passe doivent contenir douze caractères et être renouvelés tous les six mois.", "charte_it"),
    ("Matériel : un ordinateur portable est fourni à chaque nouvel embauché et renouvelé tous les quatre ans.", "charte_it"),
    ("Arrêt maladie : le salarié doit prévenir son manager le jour même et transmettre son arrêt sous 48 heures.", "reglement_rh"),
    ("Prime de cooptation : une prime de 1 000 euros est versée lorsqu'un candidat recommandé est embauché et valide sa période d'essai.", "reglement_rh"),
    ("Confidentialité : les documents clients ne doivent jamais être partagés via une messagerie personnelle.", "charte_it"),
    ("Horaires : les horaires de présence obligatoire sont de 10h à 16h, avec une pause déjeuner d'une heure.", "reglement_rh"),
]

# 1.2 — Indexation dans ChromaDB
try:
    chroma_client.delete_collection("corpus_tp1")
except Exception:
    pass
collection_tp1 = chroma_client.create_collection(
    name="corpus_tp1", embedding_function=embedding_fn, metadata={"hnsw:space": "cosine"}
)
collection_tp1.add(
    ids=[f"doc{i}" for i in range(len(corpus))],
    documents=[t for t, _ in corpus],
    metadatas=[{"source": s} for _, s in corpus],
)
print(f"{collection_tp1.count()} textes indexés.")

### 1.2 bis — Du texte aux chunks et aux embeddings

Avant de chercher, voyons ce que fait le système : **découper** un texte en chunks, puis **vectoriser** chaque chunk, c'est-à-dire le transformer en une liste de nombres (son embedding).

In [ ]:
# Découpage d'un texte long en chunks (même outil que dans le TP2)
texte_long = " ".join(t for t, _ in corpus)
splitter_demo = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks_demo = splitter_demo.split_text(texte_long)
print(f"Texte : {len(texte_long)} caractères → {len(chunks_demo)} chunks (taille max 200, chevauchement 30)")
for i, c in enumerate(chunks_demo):
    print(f"[chunk {i} · {len(c)} car.] {c}")

In [ ]:
# Vectorisation : chaque chunk devient un vecteur de nombres (son embedding)
import numpy as np

vecteurs = np.array(embedding_fn(chunks_demo))
print(f"{vecteurs.shape[0]} vecteurs de dimension {vecteurs.shape[1]}")
display(pd.DataFrame({
    "Chunk": [c[:50] + "..." for c in chunks_demo],
    "Début du vecteur": [str(v[:5].round(3)) for v in vecteurs],
}))

# Similarité cosinus entre chunks : plus la valeur est proche de 1, plus les textes sont proches
normes = vecteurs / np.linalg.norm(vecteurs, axis=1, keepdims=True)
matrice = pd.DataFrame(normes @ normes.T, index=range(len(chunks_demo)), columns=range(len(chunks_demo))).round(2)
print("Similarité cosinus entre chunks :")
display(matrice)

In [ ]:
# 1.3 — Fonction de recherche sémantique (top-k)
def rechercher(question, collection=collection_tp1, k=3):
    res = collection.query(query_texts=[question], n_results=k)
    lignes = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        lignes.append({"Similarité": round(1 - dist, 3), "Source": meta["source"], "Passage": doc})
    return pd.DataFrame(lignes)


# Trois questions qui parlent du même sujet avec des mots différents de ceux du texte
questions = [
    "Combien de jours de congés ai-je par an ?",
    "Puis-je travailler de chez moi ?",
    "Comment me faire rembourser un déjeuner avec un client ?",
]
pd.set_option("display.max_colwidth", 110)
for q in questions:
    print(f"\n❓ {q}")
    display(rechercher(q))

### 1.3 bis — Lire les scores de similarité

ChromaDB renvoie une **distance cosinus** pour chaque passage. La **similarité** affichée est `1 - distance` : elle va de 0 (aucun rapport) à 1 (sens identique). Comparez une question pertinente avec une question hors sujet.

In [ ]:
# Scores détaillés : une question pertinente, puis une question hors sujet
exemples_scores = [
    "Combien de jours de congés ai-je par an ?",
    "Quelle est la recette de la tarte aux pommes ?",
]
for q in exemples_scores:
    res = collection_tp1.query(query_texts=[q], n_results=3)
    print()
    print(f"❓ {q}")
    for rang, (doc, dist) in enumerate(zip(res["documents"][0], res["distances"][0]), start=1):
        print(f"  {rang}. distance = {dist:.3f} → similarité = {1 - dist:.3f} | {doc[:70]}...")

### 1.4 — À vous de jouer

Posez **vos propres questions** en variant la formulation (synonymes, question vague, question sans mot en commun avec le texte) et complétez la grille d'analyse ci-dessous.

In [ ]:
# Modifiez la question puis relancez la cellule
ma_question = "Que se passe-t-il si je suis malade ?"
display(rechercher(ma_question, k=3))

In [ ]:
# 1.5 — Grille d'analyse à compléter (une colonne par question)
grille_tp1 = pd.DataFrame(
    {
        "Critère": ["Top-1 pertinent ?", "Score de similarité", "Bon chunk dans le top-3 ?", "Mots communs avec la question ?"],
        "Question 1": ["", "", "", ""],
        "Question 2": ["", "", "", ""],
        "Question 3": ["", "", "", ""],
    }
)
grille_tp1

### 💬 Questions de synthèse TP1
- Sur quelles questions la recherche sémantique a-t-elle retrouvé le bon passage **sans** mot commun avec la question ?
- Que se passe-t-il pour une question hors sujet ? Le score de similarité permet-il de le détecter ?
- Quelle différence attendez-vous avec une recherche par mots-clés ?

---
## TP2 — Construire un pipeline RAG (~45 min)

**Objectif :** assembler les briques vues en cours pour créer un assistant qui répond à partir d'un **document PDF** de votre choix (rapport, procédure, règlement, etc.), en citant ses sources.

**Étape 1 — Indexer** : charger le PDF, le découper en chunks, le vectoriser et le stocker dans ChromaDB.

In [ ]:
# 2.1 — Charger un PDF (glissez un fichier depuis votre ordinateur)
from google.colab import files
from pypdf import PdfReader

uploaded = files.upload()
chemin_pdf = next(iter(uploaded))
lecteur = PdfReader(chemin_pdf)
pages = [(i + 1, (p.extract_text() or "")) for i, p in enumerate(lecteur.pages)]
print(f"{chemin_pdf} : {len(pages)} pages, {sum(len(t) for _, t in pages)} caractères")

In [ ]:
# 2.2 — Découper en chunks (avec chevauchement) et indexer
# Paramètres à expérimenter en 2.6
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100

def indexer_pdf(pages, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, nom="doc_pdf"):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    try:
        chroma_client.delete_collection(nom)
    except Exception:
        pass
    col = chroma_client.create_collection(name=nom, embedding_function=embedding_fn, metadata={"hnsw:space": "cosine"})
    ids, docs, metas = [], [], []
    for num_page, texte in pages:
        for j, chunk in enumerate(splitter.split_text(texte)):
            ids.append(f"p{num_page}_c{j}")
            docs.append(chunk)
            metas.append({"page": num_page})
    col.add(ids=ids, documents=docs, metadatas=metas)
    print(f"{col.count()} chunks indexés (taille {chunk_size}, chevauchement {chunk_overlap}).")
    return col

collection_pdf = indexer_pdf(pages)

**Étape 2 — Interroger** : vectoriser la question et récupérer les `k` passages les plus proches, avec leur page.

In [ ]:
# 2.3 — Recherche dans le document
def chercher_passages(question, collection, k=3):
    res = collection.query(query_texts=[question], n_results=k)
    return [
        {"texte": d, "page": m["page"], "similarite": round(1 - dist, 3)}
        for d, m, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])
    ]

question_test = "Quel est le sujet principal de ce document ?"  # à adapter à votre PDF
for p in chercher_passages(question_test, collection_pdf):
    print(f"[page {p['page']} · similarité {p['similarite']}] {p['texte'][:200]}...\n")

**Étape 3 — Générer** : construire le **prompt augmenté** (consigne + passages + question) et appeler le LLM.

In [ ]:
# 2.4 — Prompt augmenté et génération
CONSIGNE_SYSTEME = (
    "Tu es un assistant qui répond uniquement à partir des extraits de document fournis. "
    "Si l'information n'est pas dans les extraits, réponds exactement : "
    "« Je ne trouve pas l'information dans le document. » "
    "Cite toujours la ou les pages utilisées sous la forme [p. X]."
)

def repondre_rag(question, collection=None, k=3):
    collection = collection or collection_pdf
    passages = chercher_passages(question, collection, k)
    contexte = "\n\n".join(f"[p. {p['page']}] {p['texte']}" for p in passages)
    prompt = f"Extraits du document :\n{contexte}\n\nQuestion : {question}\nRéponse :"
    reponse = ask_model(prompt, system=CONSIGNE_SYSTEME)
    return reponse, passages

# Adaptez la question à votre document
question = "Quel est le sujet principal de ce document ?"
reponse, passages = repondre_rag(question)
print(reponse)
print("\nPages consultées :", sorted({p["page"] for p in passages}))

### 2.5 — Comparer avec et sans RAG

Posez la même question au LLM **seul** (sans les extraits) et comparez.

In [ ]:
reponse_sans_rag = ask_model(question)
print("=== SANS RAG ===\n", reponse_sans_rag)
print("\n=== AVEC RAG ===\n", reponse)

### 2.6 — Expérimenter les paramètres

Relancez l'indexation avec d'autres valeurs (`chunk_size` = 300, 800 ou 1500 ; `chunk_overlap` = 0 ou 150) et testez `k` = 1, 3 ou 6. Notez ce qui change dans les passages retrouvés et dans la réponse.

In [ ]:
# Exemple : chunks plus petits, plus de passages
collection_test = indexer_pdf(pages, chunk_size=300, chunk_overlap=50, nom="doc_pdf_test")
r, p = repondre_rag(question, collection=collection_test, k=5)
print(r)

### 💬 Questions de synthèse TP2
- Quel effet a la taille des chunks sur la pertinence des passages retrouvés ?
- Que répond l'assistant à une question dont la réponse n'est **pas** dans le document ?
- Quelles limites du pipeline avez-vous rencontrées (tableaux, mise en page, PDF scanné) ?

---
## TP3 — Évaluer le RAG (~25 min)

**Objectif :** comparer les réponses **sans RAG** et **avec RAG** sur plusieurs questions portant sur votre document, puis les noter avec une grille d'évaluation.

**Grille (1 = mauvais, 5 = excellent) :** Pertinence du retrieval · Fidélité au contexte · Clarté · Présence de sources

In [ ]:
# 3.1 — Trois questions sur VOTRE document (à compléter) :
#   1 question dont la réponse est explicite, 1 question qui demande de croiser deux passages,
#   1 question dont la réponse n'est PAS dans le document
questions_eval = [
    "",  # question 1
    "",  # question 2
    "",  # question 3 (hors document)
]

resultats = []
for q in [q for q in questions_eval if q]:
    sans = ask_model(q)
    avec, passages = repondre_rag(q)
    resultats.append({"Question": q, "Sans RAG": sans, "Avec RAG": avec, "Pages": sorted({p["page"] for p in passages})})

if resultats:
    df_eval = pd.DataFrame(resultats)
    pd.set_option("display.max_colwidth", 300)
    display(df_eval)
else:
    print("Complétez questions_eval ci-dessus puis relancez la cellule.")

In [ ]:
# 3.2 — Notez chaque approche pour une question de votre choix
grille_rag = pd.DataFrame(
    {
        "Critère": ["Pertinence du retrieval", "Fidélité au contexte", "Clarté", "Présence de sources"],
        "Sans RAG": [None, None, None, None],
        "Avec RAG": [None, None, None, None],
    }
)
# Exemple : grille_rag.loc[0, "Avec RAG"] = 4
grille_rag

In [ ]:
# 3.3 — Score total (à exécuter une fois la grille remplie)
if grille_rag[["Sans RAG", "Avec RAG"]].isnull().values.any():
    print("⚠️  Complétez d'abord la grille (remplacez les None par une note de 1 à 5).")
else:
    print("Score total sur 20 :")
    print(grille_rag[["Sans RAG", "Avec RAG"]].sum())

### 💬 Questions de synthèse TP3
- Sur quel critère le RAG apporte-t-il le plus d'amélioration ? Sur lequel apporte-t-il le moins ?
- Le modèle a-t-il respecté la consigne « je ne trouve pas l'information » pour la question hors document ?
- Comment automatiser cette évaluation à grande échelle ? (piste : LLM-as-a-judge, Séance 3)

---
## Bonus (optionnel) — Une interface de chat avec Gradio

Pour les plus rapides : transformez votre assistant en petite application web.

In [ ]:
!pip install -q gradio
import gradio as gr

def repondre_interface(question):
    reponse, passages = repondre_rag(question)
    pages_citees = ", ".join(str(p) for p in sorted({x["page"] for x in passages}))
    return f"{reponse}\n\n(Pages consultées : {pages_citees})"

demo = gr.Interface(fn=repondre_interface, inputs=gr.Textbox(label="Votre question"), outputs=gr.Textbox(label="Réponse"), title="Assistant documentaire RAG")
demo.launch()

---
## Synthèse de la séance

- Le **RAG** ancre le LLM sur vos propres documents, réduit les hallucinations et permet de citer des sources.
- Le pipeline se déroule en deux phases : **indexer** (charger, découper, vectoriser, stocker), puis **interroger** (rechercher, générer).
- La **taille des chunks**, le **chevauchement** et le **nombre de passages `k`** influencent directement la qualité.
- Le **prompt augmenté** doit imposer de répondre à partir du contexte, et autoriser le « je ne sais pas ».
- Un RAG s'évalue à deux niveaux : **le retrieval** puis **la réponse** (fidélité, clarté, sources).

### Pour la Séance 3
- Terminez ce notebook et conservez vos résultats de comparaison avec et sans RAG.
- Vérifiez que vous pouvez activer un **GPU gratuit** dans Colab (Exécution > Modifier le type d'exécution).

**Séance 3 : Fine-tuning, évaluation et projet** — Prompt engineering vs RAG vs fine-tuning, LoRA/QLoRA, évaluation → TP : fine-tuning léger et comparaison des approches.